# 🏋️ Week 4: End-to-End ML Project Practice (Enhanced)

This notebook guides you through building a complete ML project with **detailed explanations** of:
- **WHAT** each step accomplishes and why it matters
- **HOW** to structure a professional ML workflow
- **WHY** certain decisions are made
- **WHEN** to use different approaches in real projects

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ Imports ready!")

---

## 📚 The ML Project Lifecycle

### Overview of Steps

```
1. Problem Definition      → What are we trying to solve?
2. Data Collection         → Where does data come from?
3. Exploratory Data Analysis (EDA) → What does the data look like?
4. Data Preprocessing      → How do we clean the data?
5. Feature Engineering     → Can we create better features?
6. Model Selection         → Which algorithm should we use?
7. Model Training          → Train and validate the model
8. Hyperparameter Tuning   → Optimize model parameters
9. Model Evaluation        → How well does it perform?
10. Deployment             → How do we serve predictions?
```

### Time Allocation (Typical)

| Phase | Time | Notes |
|-------|------|-------|
| Understanding Problem | 10% | Most important step! |
| Data Collection & Cleaning | 30% | "Garbage in, garbage out" |
| EDA & Feature Engineering | 30% | Often makes the biggest impact |
| Modeling & Evaluation | 20% | Usually the "easy" part |
| Deployment & Monitoring | 10% | Often underestimated |

---

## 📚 Exercise 1: Problem Definition Framework

### What is Problem Definition?
Before writing any code, you must clearly understand:
- **Business Objective**: What problem are we solving?
- **Success Metrics**: How do we measure success?
- **Constraints**: Time, budget, data limitations?
- **Stakeholders**: Who will use this model?

### CRISP-DM Framework
```
1. Business Understanding → Define objectives
2. Data Understanding     → Explore and validate data
3. Data Preparation       → Clean and transform
4. Modeling               → Build and evaluate models
5. Evaluation             → Assess against business goals
6. Deployment             → Put into production
```

### Key Questions to Ask
1. **What is the target variable?** (Classification, Regression, Clustering?)
2. **What metrics matter?** (Accuracy? Precision? Revenue impact?)
3. **What's the baseline?** (Current solution, random guess?)
4. **What's the cost of errors?** (False positives vs false negatives?)
5. **How will predictions be used?** (Real-time? Batch? Human review?)

---

In [ ]:
def define_problem(problem_name: str):
    """
    Template for defining an ML problem.
    
    Fill this out before starting any project!
    """
    problem_definition = {
        'name': problem_name,
        'business_objective': 'TODO: What business problem are we solving?',
        'ml_task': 'TODO: Classification / Regression / Clustering / etc.',
        'target_variable': 'TODO: What are we predicting?',
        'success_metric': 'TODO: How do we measure success?',
        'baseline': 'TODO: What is the current solution or baseline?',
        'constraints': 'TODO: Time, cost, data limitations?',
        'stakeholders': 'TODO: Who will use this model?',
        'error_costs': {
            'false_positive': 'TODO: Cost of wrongly predicting positive',
            'false_negative': 'TODO: Cost of missing a positive'
        }
    }
    return problem_definition

# Example: Customer Churn Prediction
churn_problem = {
    'name': 'Customer Churn Prediction',
    'business_objective': 'Identify customers likely to cancel subscription so we can offer retention incentives',
    'ml_task': 'Binary Classification',
    'target_variable': 'churned (1 = canceled, 0 = retained)',
    'success_metric': 'Recall (we want to catch all potential churners)',
    'baseline': 'Currently 20% of customers churn; no prediction model exists',
    'constraints': 'Must run daily, predictions needed before morning team meeting',
    'stakeholders': 'Customer Success Team, Marketing',
    'error_costs': {
        'false_positive': '$20 (cost of unnecessary retention offer)',
        'false_negative': '$500 (lost customer lifetime value)'
    }
}

print("=" * 60)
print("PROBLEM DEFINITION")
print("=" * 60)
for key, value in churn_problem.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  - {k}: {v}")
    else:
        print(f"\n{key}:\n  {value}")

---

## 📚 Exercise 2: Exploratory Data Analysis (EDA)

### What is EDA?
**Exploratory Data Analysis** is the process of visually and statistically exploring data to:
- Understand data distributions
- Find patterns and anomalies
- Identify data quality issues
- Generate hypotheses

### EDA Checklist

#### 1. Basic Statistics
```python
df.shape          # Rows and columns
df.info()         # Data types and missing values
df.describe()     # Summary statistics
df.nunique()      # Unique values per column
```

#### 2. Missing Values
```python
df.isnull().sum()           # Count per column
df.isnull().sum() / len(df) # Percentage
```

#### 3. Distribution Analysis
- **Numerical**: Histograms, box plots, skewness
- **Categorical**: Value counts, bar plots

#### 4. Correlation Analysis
```python
df.corr()  # Correlation matrix
```

#### 5. Target Variable Analysis
- Class balance (for classification)
- Distribution (for regression)
- Relationship with features

---

In [ ]:
# Create synthetic customer data for our project
np.random.seed(42)
n_samples = 1000

# Generate data
data = {
    'customer_id': range(1, n_samples + 1),
    'age': np.random.randint(18, 70, n_samples),
    'tenure_months': np.random.exponential(24, n_samples).astype(int),
    'monthly_charges': np.random.normal(70, 20, n_samples).clip(20, 150),
    'total_charges': np.random.exponential(2000, n_samples),
    'num_support_tickets': np.random.poisson(2, n_samples),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(['Credit card', 'Bank transfer', 'Electronic check'], n_samples),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.4, 0.4, 0.2]),
}

# Create churn target (correlated with features)
churn_prob = (
    0.3 * (data['contract_type'] == 'Month-to-month') +
    0.2 * (data['num_support_tickets'] > 3) +
    0.15 * (data['tenure_months'] < 12) +
    0.1 * (data['payment_method'] == 'Electronic check')
) + np.random.normal(0, 0.1, n_samples)

data['churned'] = (churn_prob > 0.3).astype(int)

df = pd.DataFrame(data)

# Add some missing values
df.loc[np.random.choice(n_samples, 50), 'age'] = np.nan
df.loc[np.random.choice(n_samples, 30), 'total_charges'] = np.nan

print("Dataset created!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
def comprehensive_eda(df: pd.DataFrame, target_col: str):
    """
    Perform comprehensive EDA on a dataset.
    
    This function generates:
    1. Basic statistics
    2. Missing value analysis
    3. Distribution plots
    4. Correlation analysis
    5. Target variable analysis
    """
    print("=" * 60)
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 60)
    
    # 1. Basic Info
    print("\n1. BASIC STATISTICS")
    print("-" * 40)
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"\nColumn Types:")
    print(df.dtypes.value_counts())
    
    # 2. Missing Values
    print("\n2. MISSING VALUES")
    print("-" * 40)
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
    print(missing_df[missing_df['Count'] > 0])
    
    # 3. Target Variable
    print(f"\n3. TARGET VARIABLE: {target_col}")
    print("-" * 40)
    print(df[target_col].value_counts())
    print(f"\nClass Balance: {df[target_col].value_counts(normalize=True).round(3).to_dict()}")
    
    # 4. Numerical Features
    print("\n4. NUMERICAL FEATURES")
    print("-" * 40)
    num_cols = df.select_dtypes(include=[np.number]).columns.drop([target_col, 'customer_id'], errors='ignore')
    print(df[num_cols].describe().round(2))
    
    # 5. Categorical Features
    print("\n5. CATEGORICAL FEATURES")
    print("-" * 40)
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        print(f"\n{col}:")
        print(df[col].value_counts())
    
    return num_cols, cat_cols

# Run EDA
num_cols, cat_cols = comprehensive_eda(df, 'churned')

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Target distribution
df['churned'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['green', 'red'])
axes[0, 0].set_title('Target: Churned Distribution')
axes[0, 0].set_xticklabels(['Retained', 'Churned'], rotation=0)

# 2. Age distribution by churn
for churn in [0, 1]:
    axes[0, 1].hist(df[df['churned']==churn]['age'].dropna(), alpha=0.5, 
                    label=f"Churned={churn}", bins=20)
axes[0, 1].set_title('Age Distribution by Churn')
axes[0, 1].legend()

# 3. Tenure distribution by churn
for churn in [0, 1]:
    axes[0, 2].hist(df[df['churned']==churn]['tenure_months'], alpha=0.5, 
                    label=f"Churned={churn}", bins=20)
axes[0, 2].set_title('Tenure Distribution by Churn')
axes[0, 2].legend()

# 4. Contract type vs churn
churn_by_contract = df.groupby('contract_type')['churned'].mean().sort_values(ascending=False)
churn_by_contract.plot(kind='bar', ax=axes[1, 0], color='coral')
axes[1, 0].set_title('Churn Rate by Contract Type')
axes[1, 0].set_ylabel('Churn Rate')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)

# 5. Correlation heatmap
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', ax=axes[1, 1], fmt='.2f')
axes[1, 1].set_title('Numerical Feature Correlations')

# 6. Support tickets vs churn
churn_by_tickets = df.groupby('num_support_tickets')['churned'].mean()
churn_by_tickets.plot(kind='bar', ax=axes[1, 2], color='steelblue')
axes[1, 2].set_title('Churn Rate by Support Tickets')
axes[1, 2].set_ylabel('Churn Rate')

plt.tight_layout()
plt.show()

print("\n📊 KEY INSIGHTS FROM EDA:")
print("-" * 50)
print("1. Month-to-month contracts have highest churn rate")
print("2. Shorter tenure correlates with higher churn")
print("3. More support tickets → higher churn")
print("4. Class imbalance exists but is manageable")

---

## 📚 Exercise 3: Building the ML Pipeline

### Pipeline Best Practices

1. **Separation of Concerns**
   - Preprocessing separate from modeling
   - Easy to swap components

2. **Reproducibility**
   - Set random seeds
   - Version control data and models

3. **Prevent Data Leakage**
   - Fit only on training data
   - Use pipelines that enforce this

4. **Modularity**
   - Functions for each step
   - Easy to test and debug

---

In [ ]:
def build_complete_pipeline(numeric_features: list, categorical_features: list):
    """
    Build a complete preprocessing + modeling pipeline.
    
    Pipeline Structure:
    1. Numeric: Impute → Scale
    2. Categorical: Impute → One-Hot Encode
    3. Combine with ColumnTransformer
    4. Add classifier
    
    Returns:
        sklearn Pipeline object
    """
    # Numeric pipeline
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    # Categorical pipeline
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
    ])
    
    # Combine
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    # Full pipeline with model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ])
    
    return pipeline

# Define features
numeric_features = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'num_support_tickets']
categorical_features = ['contract_type', 'payment_method', 'internet_service']

# Prepare data
X = df.drop(['customer_id', 'churned'], axis=1)
y = df['churned']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nClass distribution in training:")
print(y_train.value_counts(normalize=True).round(3))

---

## 📚 Exercise 4: Model Comparison

### Why Compare Multiple Models?

Different algorithms have different strengths:

| Model | Strengths | Weaknesses | Best For |
|-------|-----------|------------|----------|
| Logistic Regression | Fast, interpretable | Linear boundaries | Baseline, regulated industries |
| Random Forest | Handles non-linearity, robust | Less interpretable | General purpose |
| Gradient Boosting | Often best performance | Slow to train | Competitions, when accuracy critical |

### Model Selection Strategy

1. **Start simple** (Logistic Regression as baseline)
2. **Add complexity** (Tree ensembles)
3. **Compare with cross-validation** (not just train/test)
4. **Consider constraints** (speed, interpretability)

---

In [ ]:
def compare_models(X_train, y_train, numeric_features, categorical_features):
    """
    Compare multiple classification models using cross-validation.
    
    Models compared:
    - Logistic Regression (baseline)
    - Random Forest
    - Gradient Boosting
    
    Returns:
        Dict of model results
    """
    # Preprocessing (same for all models)
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    # Define models to compare
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
    }
    
    results = {}
    
    print("Model Comparison (5-Fold CV)")
    print("=" * 60)
    
    for name, model in models.items():
        # Create pipeline
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        
        # Cross-validation for multiple metrics
        cv_accuracy = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
        cv_roc_auc = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc')
        cv_f1 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
        
        results[name] = {
            'accuracy_mean': cv_accuracy.mean(),
            'accuracy_std': cv_accuracy.std(),
            'roc_auc_mean': cv_roc_auc.mean(),
            'roc_auc_std': cv_roc_auc.std(),
            'f1_mean': cv_f1.mean(),
            'f1_std': cv_f1.std(),
            'pipeline': pipeline
        }
        
        print(f"\n{name}:")
        print(f"  Accuracy: {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std()*2:.4f})")
        print(f"  ROC-AUC:  {cv_roc_auc.mean():.4f} (+/- {cv_roc_auc.std()*2:.4f})")
        print(f"  F1-Score: {cv_f1.mean():.4f} (+/- {cv_f1.std()*2:.4f})")
    
    return results

# Run comparison
model_results = compare_models(X_train, y_train, numeric_features, categorical_features)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['accuracy', 'roc_auc', 'f1']
colors = ['steelblue', 'coral', 'green']

for i, metric in enumerate(metrics):
    names = list(model_results.keys())
    means = [model_results[n][f'{metric}_mean'] for n in names]
    stds = [model_results[n][f'{metric}_std'] for n in names]
    
    bars = axes[i].bar(names, means, yerr=stds, capsize=5, color=colors[i], alpha=0.7)
    axes[i].set_title(f'{metric.upper()}')
    axes[i].set_ylim(0.5, 1.0)
    axes[i].set_xticklabels(names, rotation=45, ha='right')
    
    # Add value labels
    for bar, mean in zip(bars, means):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{mean:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Select best model
best_model = max(model_results.items(), key=lambda x: x[1]['roc_auc_mean'])
print(f"\n🏆 Best Model (by ROC-AUC): {best_model[0]}")

---

## 📚 Exercise 5: Final Evaluation & Reporting

### What to Include in a Model Report

1. **Problem Statement**: What we're solving
2. **Data Summary**: Size, features, target distribution
3. **Methodology**: Preprocessing, model selection
4. **Results**: Metrics, confusion matrix, ROC curve
5. **Insights**: Feature importance, error analysis
6. **Recommendations**: Next steps, deployment considerations

---

In [ ]:
def final_evaluation(pipeline, X_train, y_train, X_test, y_test, model_name):
    """
    Perform final model evaluation and generate report.
    """
    # Fit on full training set
    pipeline.fit(X_train, y_train)
    
    # Predictions
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    print("=" * 60)
    print(f"FINAL MODEL EVALUATION: {model_name}")
    print("=" * 60)
    
    # Classification Report
    print("\n1. CLASSIFICATION REPORT")
    print("-" * 40)
    print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))
    
    # ROC-AUC
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f"\n2. ROC-AUC Score: {roc_auc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n3. CONFUSION MATRIX")
    print("-" * 40)
    print(f"               Predicted")
    print(f"              Retain  Churn")
    print(f"Actual Retain   {cm[0,0]:4d}   {cm[0,1]:4d}")
    print(f"       Churn    {cm[1,0]:4d}   {cm[1,1]:4d}")
    
    return pipeline, y_pred, y_proba

# Get best model and evaluate
best_pipeline = model_results['Random Forest']['pipeline']
trained_pipeline, y_pred, y_proba = final_evaluation(
    best_pipeline, X_train, y_train, X_test, y_test, 'Random Forest'
)

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Feature Importance
# Get feature names after preprocessing
preprocessor = trained_pipeline.named_steps['preprocessor']
classifier = trained_pipeline.named_steps['classifier']

# Get one-hot encoded feature names
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_feature_names = list(cat_encoder.get_feature_names_out(categorical_features))
all_feature_names = numeric_features + cat_feature_names

# Get importances
importances = classifier.feature_importances_
indices = np.argsort(importances)[::-1][:10]  # Top 10

axes[2].barh(range(len(indices)), importances[indices])
axes[2].set_yticks(range(len(indices)))
axes[2].set_yticklabels([all_feature_names[i] for i in indices])
axes[2].set_title('Top 10 Feature Importances')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n�� KEY FINDINGS:")
print("-" * 50)
print("1. Top predictors: tenure_months, num_support_tickets, monthly_charges")
print("2. Contract type significantly impacts churn")
print("3. Model achieves good balance between precision and recall")

---

## 📋 Week 4 Practice Summary

### End-to-End Project Checklist

- [ ] **Problem Definition**: Clear objectives and success metrics
- [ ] **Data Understanding**: EDA, missing values, distributions
- [ ] **Preprocessing Pipeline**: Reproducible, no data leakage
- [ ] **Model Selection**: Compare multiple algorithms
- [ ] **Hyperparameter Tuning**: GridSearchCV or RandomizedSearchCV
- [ ] **Final Evaluation**: Multiple metrics, confusion matrix, ROC
- [ ] **Documentation**: Code comments, findings, recommendations

### Interview Tips
1. **Start with the business problem** - not the algorithm
2. **Show EDA skills** - plots and insights matter
3. **Use pipelines** - demonstrates production thinking
4. **Explain trade-offs** - precision vs recall, complexity vs speed
5. **Know your metrics** - why you chose them

---
**Ready for Week 5: NLP & Deployment!** 🚀